# 03 — Modeling: predicting stickiness

We train interpretable models to predict the **sticky** label (top ~20% popularity) from audio features. This does **not** claim audio *causes* popularity — marketing, artist reach, and timing confound the outcome.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import features, modeling, visuals

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
TBL_DIR = PROJECT_ROOT / "outputs" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "spotify_model_data.csv"
df = pd.read_csv(DATA_PATH)
df = df.reset_index(drop=True)
print(df.shape)


## 1. Feature matrix & train/test split (80/20, stratified when possible)


In [ ]:
INCLUDE_GENRE = False

X, y = features.split_features_target(df, target_col="sticky", include_genre=INCLUDE_GENRE)
feature_names = list(X.columns)
assert X.notna().all().all(), "NaNs in feature matrix"

X_train, X_test, y_train, y_test = modeling.train_test_split_stratified(
    X, y, test_size=0.2, random_state=42
)

X_train_s, X_test_s, scaler = features.scale_train_test(X_train, X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Features:", feature_names)


## 2. Logistic regression (baseline, standardized features)


In [ ]:
log_reg = modeling.train_logistic_regression(X_train_s, y_train, class_weight='balanced')
log_metrics = modeling.evaluate_classifier(log_reg, X_test_s, y_test)
print("Logistic regression metrics:", log_metrics)

fig = visuals.plot_logistic_coefficients(
    log_reg, feature_names, save_path=FIG_DIR / "09_logistic_coefficients.png"
)
plt.show()

coef = log_reg.coef_.ravel()
order = np.argsort(coef)
print("Top negative (lower log-odds for sticky):", list(zip(np.array(feature_names)[order[:5]], coef[order[:5]])))
print("Top positive (higher log-odds for sticky):", list(zip(np.array(feature_names)[order[-5:]], coef[order[-5:]])))


## 3. Random forest


In [ ]:
rf = modeling.train_random_forest(X_train, y_train, class_weight='balanced')
rf_metrics = modeling.evaluate_classifier(rf, X_test, y_test)
print("Random forest metrics:", rf_metrics)

importances = rf.feature_importances_
fig = visuals.plot_feature_importance(
    importances,
    feature_names,
    title="Random forest feature importance",
    save_path=FIG_DIR / "10_rf_feature_importance.png",
)
plt.show()

imp_order = np.argsort(importances)
print("Top 5 RF importance:", list(zip(np.array(feature_names)[imp_order[-5:]], importances[imp_order[-5:]])))


## 4. Model comparison


In [ ]:
comp = modeling.build_classification_report_df({
    "logistic_regression": log_metrics,
    "random_forest": rf_metrics,
})
display(comp.round(4))
comp.to_csv(TBL_DIR / "model_metrics.csv")


## 5. Confusion matrices


In [ ]:
fig = visuals.plot_confusion_matrix_from_model(
    log_reg, X_test_s, y_test, save_path=FIG_DIR / "11_confusion_matrix_logistic.png"
)
plt.show()
fig = visuals.plot_confusion_matrix_from_model(
    rf, X_test, y_test, save_path=FIG_DIR / "12_confusion_matrix_rf.png"
)
plt.show()


## 5b. ROC and precision–recall curves

ROC summarizes rank discrimination; precision–recall is often more informative under **class imbalance** (~20% sticky).


In [ ]:
pos_lr = int(np.where(log_reg.classes_ == 1)[0][0])
y_score_lr = log_reg.predict_proba(X_test_s)[:, pos_lr]
pos_rf = int(np.where(rf.classes_ == 1)[0][0])
y_score_rf = rf.predict_proba(X_test)[:, pos_rf]

fig = visuals.plot_roc_comparison(
    y_test, y_score_lr, y_score_rf, save_path=FIG_DIR / "13_roc_curves.png"
)
plt.show()
fig = visuals.plot_precision_recall_comparison(
    y_test, y_score_lr, y_score_rf, save_path=FIG_DIR / "14_pr_curves.png"
)
plt.show()


## 6. Optional: regression on `popularity_z`


In [ ]:
if "popularity_z" not in df.columns:
    print("popularity_z missing — skip regression block.")
else:
    yz_train = df.loc[X_train.index, "popularity_z"]
    yz_test = df.loc[X_test.index, "popularity_z"]
    lin = modeling.train_linear_regression(X_train_s, yz_train)
    reg_metrics = modeling.evaluate_regression(lin, X_test_s, yz_test)
    print("Regression on popularity_z:", reg_metrics)


## 7. Optional: OLS on `popularity_z`


In [ ]:
if "popularity_z" not in df.columns:
    print("Skip OLS.")
else:
    y_pop = df.loc[X_train.index, "popularity_z"]
    X_ols = sm.add_constant(X_train_s, has_constant="add")
    ols = sm.OLS(y_pop, X_ols).fit()
    print(ols.summary().tables[1])


## 8. Error analysis (logistic)


In [ ]:
test_idx = X_test.index
meta_cols = [c for c in ("track_name", "artist_name", "genre", "popularity") if c in df.columns]

y_pred_lr = log_reg.predict(X_test_s)
err = pd.DataFrame({"y_true": y_test, "y_pred": y_pred_lr}, index=test_idx)
for c in meta_cols:
    err[c] = df.loc[test_idx, c].values

fp = err[(err["y_true"] == 0) & (err["y_pred"] == 1)]
fn = err[(err["y_true"] == 1) & (err["y_pred"] == 0)]
print("False positives (sample up to 5):")
display(fp.head(5))
print("False negatives (sample up to 5):")
display(fn.head(5))


## 9. Conclusions

- Compare **ROC-AUC** and **F1** between models; the comparison table is saved to `outputs/tables/model_metrics.csv`.
- **Logistic coefficients** are on log-odds scale (per standardized feature unit). **RF importance** is split-based — rank correlation, not identical ranking.
- **Limitations:** popularity proxy; confounds; correlation is not causation.
